# 04 — Evaluation & Tuning
**Project:** Customer Churn Prediction (Telco) &nbsp;|&nbsp; **Builds on:** Milestones 01–03

**Objective:** evaluate the models with metrics that are actually appropriate for an imbalanced classification problem
(not accuracy alone), then tune each model's hyperparameters with `GridSearchCV` to get the best fair comparison before
picking a final model.

Every cell in this notebook computes real output from whatever data is loaded — including the interpretation paragraphs at
the end, which are generated from the actual numbers, not written in advance. Run the whole notebook top to bottom (or
**Run All**) and everything, including the write-up, fills itself in.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import cross_val_score, train_test_split

from src.config import RANDOM_STATE, TEST_SIZE
from src.data_preparation import clean, load_raw, make_xy
from src.preprocessing import build_logistic_class_weighted, build_random_forest_class_weighted
from src.tuning import PARAM_GRIDS, build_grid_search

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
print("Setup complete. Random seed:", RANDOM_STATE)

Setup complete. Random seed: 42


## 1. Recap: load, clean, split
Same seed and logic as Milestones 01–03.

In [2]:
raw = load_raw()
df, cleaning_log = clean(raw)
X, y = make_xy(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "| Test:", X_test.shape, "| Churn rate (train):", round(y_train.mean(), 3))

Train: (5634, 19) | Test: (1409, 19) | Churn rate (train): 0.265


## 2. Why accuracy is not the right metric here

In [3]:
print(f"If a model predicted 'stays' for every customer, it would be right {(1 - y_train.mean()):.1%} of the time")
print("...while never once identifying an actual churner (recall = 0, precision = undefined).")
print("This is exactly what the Milestone 01 dummy classifier did — see reports/baseline_metrics.json.")

If a model predicted 'stays' for every customer, it would be right 73.5% of the time
...while never once identifying an actual churner (recall = 0, precision = undefined).
This is exactly what the Milestone 01 dummy classifier did — see reports/baseline_metrics.json.


Accuracy rewards a model for being right on the *majority* class, which here is "stays." A model can score high on
accuracy while being completely useless at the one thing this project needs: finding likely churners. That is why this
notebook leads with `classification_report`, `confusion_matrix`, and ROC-AUC instead.

## 3. Baseline cross-validation (before tuning)
Using `cross_val_score` directly, on the two class-weighted models carried forward from Milestone 03, scored on ROC-AUC
(a threshold-independent metric appropriate for an imbalanced target).

In [4]:
from sklearn.model_selection import cross_val_score

candidates = {
    "logistic_regression": build_logistic_class_weighted(X_train),
    "random_forest": build_random_forest_class_weighted(X_train),
}

baseline_cv = {}
for name, pipeline in candidates.items():
    scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="roc_auc")
    baseline_cv[name] = scores
    print(f"{name:20s} ROC-AUC: {scores.mean():.4f} +/- {scores.std():.4f}  (folds: {np.round(scores, 3)})")

logistic_regression  ROC-AUC: 0.8454 +/- 0.0141  (folds: [0.864 0.852 0.851 0.835 0.824])
random_forest        ROC-AUC: 0.8478 +/- 0.0131  (folds: [0.864 0.858 0.85  0.841 0.827])


In [ ]:
tuned = {}
for name, pipeline in candidates.items():
    print(f"Running GridSearchCV for {name} ...")
    search = build_grid_search(pipeline, model_key=name, scoring="f1")
    search.fit(X_train, y_train)
    tuned[name] = search
    print(f"  best params: {search.best_params_}")
    print(f"  best CV F1 : {search.best_score_:.4f}\n")

In [ ]:
grid_comparison = pd.DataFrame({
    name: {"best_cv_f1": search.best_score_, "best_params": search.best_params_}
    for name, search in tuned.items()
}).T
grid_comparison

## 5. Select the final model

In [ ]:
best_name = max(tuned, key=lambda k: tuned[k].best_score_)
best_search = tuned[best_name]
best_model = best_search.best_estimator_

print(f"Selected model: {best_name}")
print(f"Selection rule: highest GridSearchCV cross-validated F1 among the tuned candidates")
print(f"Best CV F1: {best_search.best_score_:.4f}")
print(f"Best params: {best_search.best_params_}")

## 6. Final evaluation on the held-out test set
Evaluated **once**, using the exact metrics specified for a classification problem: `classification_report`,
`confusion_matrix`, `roc_auc_score`, and a manually plotted ROC curve.

In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Stayed", "Churned"], zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Stayed", "Churned"]).plot(
    ax=ax, cmap="Blues", colorbar=False
)
ax.set_title(f"Confusion matrix — {best_name} (tuned, test set)")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True negatives (correctly predicted 'stays'):  {tn}")
print(f"False positives (wrongly flagged as churn):     {fp}")
print(f"False negatives (missed churners):               {fn}")
print(f"True positives (correctly caught churners):      {tp}")

In [ ]:
auc = roc_auc_score(y_test, y_proba)
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

plt.figure(figsize=(5.5, 4.5))
plt.plot(fpr, tpr, label=f"{best_name} (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curve — tuned model, test set")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

print(f"ROC-AUC: {auc:.4f}")